# Python (Pandas) - Testovací cvičení (Loans Dataset)

V tomto sešitu plníme kroky analýzy a předzpracování datasetu `loans.csv`:
1. Načtení dat pomocí Pandas
2. Přejmenování sloupců na malá písmena
3. Zobrazení prvních 10 pozorování
4. Kontrola a konverze datových typů
5. Kontrola chybějících hodnot pomocí `.isnull()`
6. Nahrazení chybějících hodnot u kategoriálních proměnných nejčastější hodnotou (`.mode()`)
7. Výběr schválených půjček (`loans_granted`)
8. Výběr schválených půjček samostatným žadatelům (`standalone_borrowers`)
9. Agregace podle pohlaví (`loans_count_by_gender`)

## 1. Načtení knihovny Pandas a datasetu loans.csv

In [ ]:
import pandas as pd
import os

# Načtení datasetu
data_path = os.path.join("data", "loans.csv") if os.path.exists(os.path.join("data", "loans.csv")) else os.path.join("00_Prework", "data", "loans.csv")
df = pd.read_csv(data_path)
print(f"Rozměry datasetu: {df.shape[0]} řádků, {df.shape[1]} sloupců")

## 2. Přejmenování sloupců na malá písmena

In [ ]:
df.columns = df.columns.str.lower()
print("Sloupce:", df.columns.tolist())

## 3. Zobrazení prvních 10 řádků

In [ ]:
df.head(10)

## 4. Kontrola a konverze datových typů

In [ ]:
# Výpis původních datových typů
print(df.dtypes)

# Konverze kategoriálních proměnných na category
cat_cols = ['gender', 'married', 'education', 'self_employed', 'area', 'status']
for col in cat_cols:
    df[col] = df[col].astype('category')

df.info()

## 5. Počet chybějících hodnot (.isnull())

In [ ]:
df.isnull().sum()

## 6. Nahrazení chybějících hodnot u kategoriálních proměnných nejčastější hodnotou (.mode())

In [ ]:
categorical_to_impute = ['gender', 'married', 'dependents', 'self_employed']

for col in categorical_to_impute:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"Sloupec '{col}' doplněn módem: '{mode_val}'")

# Doplnění módu i pro credit_history (binární ukazatel 0/1)
df['credit_history'] = df['credit_history'].fillna(df['credit_history'].mode()[0])

# Kontrola
df.isnull().sum()

## 7. loans_granted – schválené půjčky (status == 'Y')

In [ ]:
loans_granted = df[df['status'] == 'Y'].copy()
print(f"Počet schválených půjček: {len(loans_granted)}")
loans_granted.head()

## 8. standalone_borrowers – schválené půjčky samostatným žadatelům
Samostatný žadatel (standalone applicant) žádá o půjčku sám, bez spolužadatele (`coapplicant_income == 0`).

In [ ]:
standalone_borrowers = loans_granted[loans_granted['coapplicant_income'] == 0].copy()
print(f"Počet schválených půjček samostatným žadatelům: {len(standalone_borrowers)}")
standalone_borrowers.head()

## 9. loans_count_by_gender – statistiky podle pohlaví
Spočítáme:
- počet půjček podle pohlaví
- průměrný příjem žadatele (`applicant_income`)
- průměrnou výši půjčky (`loan_amount`)

In [ ]:
loans_count_by_gender = loans_granted.groupby('gender', observed=False).agg(
    loans_count=('loan_amount', 'count'),
    avg_applicant_income=('applicant_income', 'mean'),
    avg_loan_amount=('loan_amount', 'mean')
).reset_index()

loans_count_by_gender['avg_applicant_income'] = loans_count_by_gender['avg_applicant_income'].round(2)
loans_count_by_gender['avg_loan_amount'] = loans_count_by_gender['avg_loan_amount'].round(2)

loans_count_by_gender